## QQB Dataset Exploration

In [1]:
import h5py
import numpy as np

path = r"C:\Users\zanca\OneDrive\Desktop\Vrij Unversiteit\extra_year\Thesis\Rapid Assessment of Earthquake Building Damage\0_data_preprocessing\qqb_dataset\QQB_earthquake_building_dataset\earthquake_building_dataset\intact\1139262234_optftp.mat"

with h5py.File(path, 'r') as f:
    print("Keys:", list(f.keys()))
    for key in f.keys():
        data = f[key][:]
        print(f"{key}: shape={data.shape}, dtype={data.dtype}, min={data.min():.4f}, max={data.max():.4f}")

Keys: ['x4']
x4: shape=(100, 99), dtype=uint8, min=0.0000, max=1.0000


In [ ]:
import os
import h5py
from scipy.io import loadmat
import pandas as pd
from sklearn.model_selection import train_test_split

***1. Exploration of the dataset:***

In [33]:

qqb_damaged_dir = r"C:\Users\zanca\OneDrive\Desktop\Vrij Unversiteit\extra_year\Thesis\Rapid Assessment of Earthquake Building Damage\0_data_preprocessing\qqb_dataset\QQB_earthquake_building_dataset\earthquake_building_dataset\damaged"
qqb_intact_dir = r"C:\Users\zanca\OneDrive\Desktop\Vrij Unversiteit\extra_year\Thesis\Rapid Assessment of Earthquake Building Damage\0_data_preprocessing\qqb_dataset\QQB_earthquake_building_dataset\earthquake_building_dataset\intact"


def get_opt_files(folder_path: str):
    files = []
    for file_name in os.listdir(folder_path):
        parts = file_name.split("_")
        if len(parts) <= 1:
            continue

        modality = parts[1].split(".")[0]
        if modality == "opt":   # keep only optical for now
            files.append(file_name)

    return files


damaged_opt_files = get_opt_files(qqb_damaged_dir)
intact_opt_files = get_opt_files(qqb_intact_dir)

print("Damaged optical files:", len(damaged_opt_files))
print("Intact optical files:", len(intact_opt_files))


def inspect_h5_file(file_path: str):
    with h5py.File(file_path, "r") as f:
        print("Keys:", list(f.keys()))

        for key in f.keys():
            data = f[key]
            print(f"\nKey: {key}")
            print("Shape:", data.shape)
            print("Type:", data.dtype)

            # Try to read small values
            arr = data[:]

            print("Min:", arr.min())
            print("Max:", arr.max())
            print("Values:", list(set(arr.flatten()[:100])))


# inspect one example from each class
if damaged_opt_files:
    print("\nDamaged example:")
    inspect_h5_file(os.path.join(qqb_damaged_dir, damaged_opt_files[0]))

if intact_opt_files:
    print("\nIntact example:")
    inspect_h5_file(os.path.join(qqb_intact_dir, intact_opt_files[0]))




Damaged optical files: 169
Intact optical files: 3860

Damaged example:
Keys: ['x3']

Key: x3
Shape: (3, 130, 95)
Type: uint8
Min: 0
Max: 255
Values: [131, 136, 140, 142, 143, 144, 145, 146, 151, 152, 153, 154, 27, 155, 157, 158, 159, 160, 161, 34, 162, 163, 165, 164, 167, 166, 171, 172, 45, 174, 177, 178, 179, 52, 50, 55, 183, 184, 59, 187, 61, 190, 189, 191, 195, 69, 71, 72, 73, 74, 79, 80, 81, 83, 84, 86, 92, 93, 95, 97, 98, 100, 102, 107, 108, 109, 110, 111, 116, 124]

Intact example:
Keys: ['x3']

Key: x3
Shape: (3, 86, 162)
Type: uint8
Min: 0
Max: 236
Values: [128, 135, 11, 140, 142, 17, 18, 19, 20, 148, 22, 23, 24, 150, 26, 27, 28, 29, 158, 31, 32, 157, 162, 156, 36, 37, 164, 159, 35, 33, 40, 42, 44, 45, 46, 38, 48, 49, 43, 51, 52, 53, 54, 59, 187, 60, 62, 63, 69, 72, 78, 80, 87, 89, 90, 91, 92, 97, 104, 110, 30, 120, 124, 126]


***2. Data cleaning and preprocessing:***

    - Extracting relevant files
    - Creating a structured format for analysis


In [ ]:

rows = []

# Damaged = 1
for fname in os.listdir(qqb_damaged_dir):
    parts = fname.split("_")
    if len(parts) >= 2 and parts[1].split(".")[0] == "opt":
        rows.append({
            "path": os.path.join(qqb_damaged_dir, fname),
            "label": 1
        })

# Intact = 0
for fname in os.listdir(qqb_intact_dir):
    parts = fname.split("_")
    if len(parts) >= 2 and parts[1].split(".")[0] == "opt":
        rows.append({
            "path": os.path.join(qqb_intact_dir, fname),
            "label": 0
        })

df = pd.DataFrame(rows)

print(df.head())
print("Total samples:", len(df))
print(df["label"].value_counts())

                                                path  label
0  C:\Users\zanca\OneDrive\Desktop\Vrij Unversite...      1
1  C:\Users\zanca\OneDrive\Desktop\Vrij Unversite...      1
2  C:\Users\zanca\OneDrive\Desktop\Vrij Unversite...      1
3  C:\Users\zanca\OneDrive\Desktop\Vrij Unversite...      1
4  C:\Users\zanca\OneDrive\Desktop\Vrij Unversite...      1
Total samples: 4029
label
0    3860
1     169
Name: count, dtype: int64


***3. Create a CSV file from the extracted data:***

    - Populte the CSV file wit the exctracted data and the corresponding labels

In [ ]:
csv_path = r"C:\Users\zanca\OneDrive\Desktop\Vrij Unversiteit\extra_year\Thesis\Rapid Assessment of Earthquake Building Damage\0_data_preprocessing\qqb_dataset\QQB_earthquake_building_dataset\earthquake_building_datasetfinal"
df.to_csv(csv_path, index=False)

***4. Dataset shuffling and splitting:***

    - Shuffle the dataset to ensure randomness
    - Split the dataset into training and validation sets

In [ ]:

# Shuffling

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Splitting

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

train_df.to_csv("qqb_train.csv", index=False)
val_df.to_csv("qqb_val.csv", index=False)